<a href="https://colab.research.google.com/github/enslern/The-AI-engineer-course-2026-Complete-AI-bootcamp/blob/main/langchain_expression_language.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PIPING A PROMPT, MODEL AND AN OUTPUT PARSER

In [69]:
!pip install langchain

In [70]:
!pip install langchain-groq


In [71]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

In [72]:
list_instructions=CommaSeparatedListOutputParser().get_format_instructions()

In [73]:
list_instructions

'Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

In [74]:
chat_template=ChatPromptTemplate([("human","ive recently adopted a {pet}. Could you suggst 3 {pet} names?"+list_instructions)])

In [75]:
chat_template.messages[0].prompt.template


'ive recently adopted a {pet}. Could you suggst 3 {pet} names?Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

In [76]:
import os
import getpass

if "GROQ_API_KEY" not in os.environ:
  os.environ["GROQ_API_KEY"]=getpass.getpass("enter the key")

In [77]:
llm=ChatGroq(
    model="qwen/qwen3.6-27b",
    temperature=0,
    max_tokens=500,
    seed=365,
    reasoning_format="hidden",
    reasoning_effort="none"
)

/usr/local/lib/python3.13/dist-packages/pydantic/main.py:263: UserWarning: WARNING! seed is not default parameter.
                    seed was transferred to model_kwargs.
                    Please confirm that seed is what you intended.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


In [78]:
list_out_parser=CommaSeparatedListOutputParser()

In [79]:
chat_template_result=chat_template.invoke({"pet":"dog"})

In [80]:
chat_template

ChatPromptTemplate(input_variables=['pet'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['pet'], input_types={}, partial_variables={}, template='ive recently adopted a {pet}. Could you suggst 3 {pet} names?Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'), additional_kwargs={})])

In [81]:
chat_result=llm.invoke(chat_template_result)

In [82]:
list_out_parser.invoke(chat_result)

['Buddy', 'Luna', 'Max']

In [83]:
chat=chat_template | llm | list_out_parser

In [84]:
chat.invoke({"pet":"cat"})

['Luna', 'Milo', 'Bella']

BATCHING

In [85]:
chat_template=ChatPromptTemplate.from_messages([("human","Ive recently adopted a {pet} which is a {breed} .Could you suggest several training tips?")])

In [86]:
chat_template

ChatPromptTemplate(input_variables=['breed', 'pet'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['breed', 'pet'], input_types={}, partial_variables={}, template='Ive recently adopted a {pet} which is a {breed} .Could you suggest several training tips?'), additional_kwargs={})])

In [87]:
chain= chat_template | llm

In [88]:
chain.invoke({"pet":"cat","breed":"persian"})

AIMessage(content='Congratulations on adopting your new Persian cat! Persians are known for their sweet, gentle, and calm personalities, but they also have specific needs due to their long coats, flat faces (brachycephalic structure), and sometimes stubborn nature.\n\nHere are several tailored training and care tips to help you and your new companion bond and thrive:\n\n### 1. **Grooming Training (Essential for Persians)**\nPersians have long, dense coats that mat easily. Training them to accept grooming is crucial.\n- **Start Early:** Begin brushing your cat daily for just 2–3 minutes. Use a wide-toothed comb or slicker brush.\n- **Positive Reinforcement:** Give treats and praise during and after grooming sessions so they associate it with positive experiences.\n- **Focus on Problem Areas:** Pay special attention to behind the ears, under the legs, and around the tail, where mats form most easily.\n- **Face Cleaning:** Due to their flat faces, tear stains can accumulate. Gently wipe t

In [89]:
%%time
chat=chain.batch([{"pet":"dog","breed":"german sheperd"},{"pet":"cat","breed":"persian"}])

CPU times: user 21.1 ms, sys: 1.12 ms, total: 22.2 ms
Wall time: 1.38 s


  STREAMING

In [90]:
response=chain.stream({"pet":"dog","breed":"german sheperd"})

In [92]:
for i in response:
  print(i.content,end=" ")

RUNNABLE AND RUNNABLE SEQUENCE CLASSES

In [93]:
type(chain)

langchain_core.runnables.base.RunnableSequence

PIPING CHAINS AND RUNNABLE PASSTHROUGH

In [95]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [94]:
chat_template_tools=ChatPromptTemplate.from_template("""what are the five most important tools a { job title} needs?
Answer only by listing the tools""")

chat_template_strategy=ChatPromptTemplate.from_template("Considering the tools provided , develop a strategy for effectively mastering these {tools}")

In [96]:
string_parser=StrOutputParser()

In [97]:
chain_tools=chat_template_tools | llm | string_parser | {"tools":RunnablePassthrough()}
chain_strategy=chat_template_strategy | llm | string_parser

In [98]:
print(chain_tools.invoke({" job title":"data scientist"}))

{'tools': '1. Python\n2. SQL\n3. Jupyter Notebook\n4. Git\n5. Pandas'}


In [99]:
print(chain_strategy.invoke({"tools":'''
1. Python
2. SQL
3. Jupyter Notebook
4. Git
5. Pandas

'''}))

To effectively master these five tools, you should not learn them in isolation. Instead, treat them as an integrated **Data Science/Engineering Workflow**. Each tool plays a specific role in the lifecycle of a data project.

Here is a strategic, phased approach to mastering them together:

### 🧩 The Integrated Workflow Mindset
Before diving into phases, understand how they connect:
1.  **SQL**: Extracts data from databases.
2.  **Python + Pandas**: Cleans, transforms, and analyzes that data.
3.  **Jupyter Notebook**: The environment where you write, test, and visualize the Python/Pandas code.
4.  **Git**: Tracks changes to your code, notebooks, and scripts for version control and collaboration.

---

## 📅 Phase 1: Foundation – Data Extraction & Basic Manipulation (Weeks 1–3)
**Goal:** Get data out of a database and perform basic operations.

### 1. SQL (The Source)
- **Focus:** `SELECT`, `WHERE`, `GROUP BY`, `JOINs`, and subqueries.
- **Practice:** Install a local database (e.g., Postg

In [100]:
chain_combined=chain_tools | chain_strategy

In [101]:
print(chain_combined.invoke({" job title":"data scientist"}))

To effectively master these five tools, you should not learn them in isolation. Instead, adopt a **project-based, integrated learning strategy**. These tools form the core stack for Data Analysis and Data Engineering.

Here’s a structured, phased strategy to master them efficiently:

### 🎯 Core Philosophy: "Learn by Building"
Don’t just read documentation. Build small projects that combine multiple tools. Each phase below introduces one new tool while reinforcing previous ones.

---

## 📅 Phase 1: Foundation – Python & Jupyter Notebook (Weeks 1–2)
**Goal:** Write clean Python code and use Jupyter for exploratory analysis.

### Key Concepts:
- **Python:** Variables, data types, loops, functions, list comprehensions, error handling.
- **Jupyter Notebook:** Cells (code vs. markdown), magic commands (`%time`, `%matplotlib inline`), saving/exporting notebooks.

### ✅ Practice Project:
**"Personal Expense Tracker"**
- Write a Python script that takes user input for expenses.
- Use Jupyter to

GRAPHING RUNNABLES

In [102]:
!pip install grandalf

In [103]:
chain_long=chat_template_tools | llm | string_parser | {"tools":RunnablePassthrough} | chat_template_strategy | llm | string_parser

In [ ]:
chain_long.get_graph().print_ascii()

RUNNABLE PARALLEL

In [104]:
from langchain_core.runnables import RunnableParallel

In [105]:
chat_template_books=ChatPromptTemplate.from_template('''
  Suggest three of the best intermediate {programming language} books.
  Answer only by listing the books.
  ''')
chat_template_projects=ChatPromptTemplate.from_template('''
  Suggest three interesting {programming language} projects for intermediate level programmers.
  Answer only by listing the projects.
  ''')

In [106]:
chain_books=chat_template_books | llm | string_parser
chain_projects =chat_template_projects | llm  | string_parser

In [108]:
chat_parallel=RunnableParallel({"books":chain_books,"projects":chain_projects})

In [107]:
print(chat_parallel.invoke({"programming language":"Python"}))

{'books': '*   *Fluent Python* by Luciano Ramalho\n*   *Effective Python* by Brett Slatkin\n*   *Python Cookbook* by David Beazley and Brian K. Jones', 'projects': '1. A real-time collaborative whiteboard application using WebSockets and Flask\n2. A personal finance tracker with data visualization using Pandas and Matplotlib\n3. An automated web scraper that monitors price changes and sends email alerts'}


In [109]:
chat_parallel.get_graph().print_ascii()

            +-------------------------------+            
            | Parallel<books,projects>Input |            
            +-------------------------------+            
                   **               **                   
                ***                   ***                
              **                         **              
+--------------------+            +--------------------+ 
| ChatPromptTemplate |            | ChatPromptTemplate | 
+--------------------+            +--------------------+ 
           *                                 *           
           *                                 *           
           *                                 *           
     +----------+                      +----------+      
     | ChatGroq |                      | ChatGroq |      
     +----------+                      +----------+      
           *                                 *           
           *                                 *           
           *  

  PIPING A RUNNABLE PARALLEL WITH OTHER RUNNABLES

In [110]:
chat_template_time=ChatPromptTemplate.from_template('''
im an intermediate level programmer
consider the following literature {books}
also consider the following projects {projects}
''')

In [111]:
chain_time=(RunnableParallel({"books":chain_books,"projects":chain_projects})
            | chat_template_time
            | llm
            | string_parser)

In [112]:
print(chain_time.invoke({"programming language":"Python"}))

This is an excellent combination of resources and projects for an intermediate Python programmer. You have selected books that move beyond syntax into **idiomatic Python, design patterns, and practical problem-solving**, paired with projects that cover **web development, data science, and automation**—three of the most common professional domains.

Here’s a structured roadmap to help you maximize your learning by integrating these books with your projects.

---

### 📚 Book-to-Project Mapping

#### 1. *Fluent Python* by Luciano Ramalho
**Focus:** Deep understanding of Python’s data model, concurrency, and idiomatic code.
**Best Paired With:** Project 1 (Real-time Collaborative Whiteboard)

- **Why?** This project involves WebSockets, which are inherently concurrent and stateful. *Fluent Python*’s chapters on **concurrency** (threads, processes, async/await) and **data structures** (dicts, sets, lists) are critical here.
- **Key Concepts to Apply:**
  - Use `asyncio` or `gevent` for hand

In [113]:
chain_time.get_graph().print_ascii()

            +-------------------------------+            
            | Parallel<books,projects>Input |            
            +-------------------------------+            
                   **               **                   
                ***                   ***                
              **                         **              
+--------------------+            +--------------------+ 
| ChatPromptTemplate |            | ChatPromptTemplate | 
+--------------------+            +--------------------+ 
           *                                 *           
           *                                 *           
           *                                 *           
     +----------+                      +----------+      
     | ChatGroq |                      | ChatGroq |      
     +----------+                      +----------+      
           *                                 *           
           *                                 *           
           *  

RUNNABLE LAMBDA

In [114]:
from langchain_core.runnables import RunnableLambda

In [115]:
runnable_sum=RunnableLambda(lambda x:sum(x))

In [116]:
runnable_sum.invoke([1,2,3,3])

9

In [117]:
runnable_sq=RunnableLambda(lambda x: x**2)

In [118]:
runnable_sq.invoke(8)

64

In [119]:
chain= runnable_sum | runnable_sq

In [121]:
chain.invoke([1,2,3])

36

In [120]:
chain.get_graph().print_ascii()

+-------------+  
| LambdaInput |  
+-------------+  
        *        
        *        
        *        
   +--------+    
   | Lambda |    
   +--------+    
        *        
        *        
        *        
   +--------+    
   | Lambda |    
   +--------+    
        *        
        *        
        *        
+--------------+ 
| LambdaOutput | 
+--------------+ 


THE @CHAIN DECORATOR

In [126]:
from langchain_core.runnables import RunnableLambda,chain

In [129]:
@chain
def runnable_sum(x):
  return sum(x)
@chain
def runnable_sq(x):
  return x**2

In [131]:
type(runnable_sum)
type(runnable_sq)

langchain_core.runnables.base.RunnableLambda

In [132]:
chain=runnable_sum | runnable_sq

In [133]:
chain.invoke([1,2,3])

36